<a href="https://colab.research.google.com/github/janepium/deteccion-anomalias-rt-iot/blob/main/notebooks/10_analisis_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Carga del dataset
2. Preparación de y_bin
3. Preparación de X
4. Train/Test
5. Random Forest inicial
6. Guardar las métricas del Random forest inicial
7. Análisis de errores

#Cargar el Dataset

In [13]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

#Cargar el archivo dataset_limpio.csv
df = pd.read_csv("/content/dataset_limpio.csv", index_col=0)

print(df.shape)

(117922, 37)


#Crear y_bin
usando la misma definicion de edwin

In [14]:
NORMAL_CLASSES = {
    "MQTT_Publish",
    "Thing_Speak",
    "Wipro_bulb"
}

y_bin = (
    ~df["Attack_type"].isin(NORMAL_CLASSES)
).astype(int)

In [15]:
print(y_bin.value_counts())
print(y_bin.value_counts(normalize=True) * 100)

Attack_type
1    105907
0     12015
Name: count, dtype: int64
Attack_type
1    89.811062
0    10.188938
Name: proportion, dtype: float64


#Preparar X

In [16]:
LEAKY = [
    "id.orig_p",
    "id.resp_p",
    "proto",
    "service"
]

X = df.drop(
    columns=["Attack_type"]
)

X = X.drop(
    columns=LEAKY
)

#Train/Test

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_bin,
    test_size=0.20,
    stratify=y_bin,
    random_state=42
)



#Random Forest inicial



In [18]:
rf_inicial = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [19]:
num_ejecuciones = 20
tiempos_entrenamiento = []

inicio_total = time.perf_counter()

for _ in range(num_ejecuciones):
    inicio = time.perf_counter()
    rf_inicial.fit(X_train, y_train)
    tiempos_entrenamiento.append(time.perf_counter() - inicio)

tiempo_total = time.perf_counter() - inicio_total
rf_inicial_train_time = sum(tiempos_entrenamiento) / num_ejecuciones

print(f"Tiempo promedio de entrenamiento ({num_ejecuciones} ejecuciones): {rf_inicial_train_time:.6f} segundos")
print(f"Tiempo total transcurrido: {tiempo_total:.2f} segundos")

Tiempo promedio de entrenamiento (20 ejecuciones): 3.648247 segundos
Tiempo total transcurrido: 72.97 segundos


In [20]:
num_ejecuciones = 20
tiempos_prediccion = []

for _ in range(num_ejecuciones):
    inicio = time.perf_counter()
    y_pred_inicial = rf_inicial.predict(X_test)
    tiempos_prediccion.append(time.perf_counter() - inicio)

# Tiempo promedio por ejecución
rf_inicial_predict_time = sum(tiempos_prediccion) / num_ejecuciones

print(
    f"Tiempo promedio de predicción ({num_ejecuciones} ejecuciones): {rf_inicial_predict_time:.6f} s")

Tiempo promedio de predicción (20 ejecuciones): 0.093708 s


In [21]:
y_prob_inicial = rf_inicial.predict_proba(X_test)[:, 1]

#Guardar las métricas del Random forest inicial

In [22]:
rf_inicial_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_inicial),
    "Precision": precision_score(
        y_test,
        y_pred_inicial,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        y_pred_inicial,
        zero_division=0
    ),
    "F1-score": f1_score(
        y_test,
        y_pred_inicial,
        zero_division=0
    ),
    "F1-Macro": f1_score(
        y_test,
        y_pred_inicial,
        average="macro",
        zero_division=0
    ),
    "PR-AUC": average_precision_score(
        y_test,
        y_prob_inicial
    ),
    "Train time": rf_inicial_train_time,
    "Predict time": rf_inicial_predict_time
}

rf_inicial_metrics

{'Accuracy': 0.9985160059359762,
 'Precision': 0.9988676574663836,
 'Recall': 0.9994806911528656,
 'F1-score': 0.9991740802793968,
 'F1-Macro': 0.9959358822010378,
 'PR-AUC': np.float64(0.9997540495539641),
 'Train time': 3.6482466438999723,
 'Predict time': 0.09370789179999975}

# Análisis de errores

In [23]:
# 1. CARGA DE DATOS Y PREPARACIÓN (Basado en la semana 4)

df = pd.read_csv("/content/dataset_limpio.csv", index_col=0)

# Definir tráfico normal y crear variable objetivo binaria
NORMAL_CLASSES = {"MQTT_Publish", "Thing_Speak", "Wipro_bulb"}
y_bin = (~df["Attack_type"].isin(NORMAL_CLASSES)).astype(int)

# Eliminar variables que generan data leakage
LEAKY = ["id.orig_p", "id.resp_p", "proto", "service"]
X = df.drop(columns=["Attack_type"] + LEAKY)

# Train/Test Split (Corregido para incluir 'Attack_type' y evitar problemas de índices)
X_train, X_test, y_train, y_test, attack_train, attack_test = train_test_split(
    X,
    y_bin,
    df["Attack_type"],
    test_size=0.20,
    stratify=y_bin,
    random_state=42
)

# 2. ENTRENAMIENTO DEL MODELO Y PREDICCIONES

rf_inicial = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_inicial.fit(X_train, y_train)

# Predicciones
y_pred = rf_inicial.predict(X_test)

# Crear DataFrame de resultados usando .values para evitar desalineación de índices
resultados = pd.DataFrame({
    'y_true_bin': y_test.values,
    'y_pred_bin': y_pred,
    'Attack_type': attack_test.values
})


# 3. TAREA 1: ANALIZAR LOS FALSOS NEGATIVOS (FN)

print("--- 1. Matriz de Confusión Global ---")
print(confusion_matrix(y_test, y_pred))

# Filtrar solo los ataques (y_true_bin == 1)
ataques_df = resultados[resultados['y_true_bin'] == 1]

# Filtrar los Falsos Negativos (Ataque real clasificado como Normal)
fn_df = ataques_df[ataques_df['y_pred_bin'] == 0]

print("\n--- Tipos de ataque en Falsos Negativos (FN) ---")
fn_counts = fn_df['Attack_type'].value_counts()
ataques_totales = ataques_df['Attack_type'].value_counts()
fn_porcentaje = (fn_counts / ataques_totales * 100).fillna(0).round(2)

analisis_fn = pd.DataFrame({
    'Total_Casos': ataques_totales,
    'Casos_FN': fn_counts.fillna(0).astype(int),
    'Porcentaje_FN (%)': fn_porcentaje
}).sort_values(by='Porcentaje_FN (%)', ascending=False)

# Mostrar solo los ataques que tienen al menos 1 FN
display(analisis_fn[analisis_fn['Casos_FN'] > 0])


# 4. TAREA 2: ANALIZAR ERRORES POR ATTACK_TYPE

print("\n--- 2. Rendimiento detallado por clase de ataque ---")

# Generar tabla con Recall por cada clase de ataque
tabla_clases = pd.DataFrame({
    'Total_Datos': ataques_totales,
    'Clasificados_Correctamente': ataques_totales - fn_counts.fillna(0),
    'Falsos_Negativos': fn_counts.fillna(0).astype(int),
    'Recall (%)': ((ataques_totales - fn_counts.fillna(0)) / ataques_totales * 100).round(2)
}).sort_values(by='Recall (%)')

display(tabla_clases)


# 5. TASK 3: ANALIZAR FALSOS POSITIVOS (FP)

# Filtrar solo el tráfico normal (y_true_bin == 0)
normal_df = resultados[resultados['y_true_bin'] == 0]
# Filtrar los Falsos Positivos (Normal real clasificado como Ataque)
fp_df = normal_df[normal_df['y_pred_bin'] == 1]

print("\n--- 3. Tráfico Normal clasificado como Ataque (FP) ---")
fp_counts = fp_df['Attack_type'].value_counts()
normal_totales = normal_df['Attack_type'].value_counts()

analisis_fp = pd.DataFrame({
    'Total_Casos': normal_totales,
    'Casos_FP': fp_counts.fillna(0).astype(int),
    'Porcentaje_FP (%)': (fp_counts / normal_totales * 100).fillna(0).round(2)
}).sort_values(by='Casos_FP', ascending=False)

display(analisis_fp)


# 6. TASK 4: RELACIÓN DE ERRORES CON CLASES MINORITARIAS

print("\n--- 4. Distribución general de clases (Muestra completa) ---")
print("Las 5 clases con MENOS datos en todo el dataset:")
print(df['Attack_type'].value_counts(ascending=True).head(5))

--- 1. Matriz de Confusión Global ---
[[ 2379    24]
 [   11 21171]]

--- Tipos de ataque en Falsos Negativos (FN) ---


,Total_Casos,Casos_FN,Porcentaje_FN (%)
Attack_type,,,
ARP_poisioning,1542,10.0,0.65
NMAP_UDP_SCAN,551,1.0,0.18



--- 2. Rendimiento detallado por clase de ataque ---


,Total_Datos,Clasificados_Correctamente,Falsos_Negativos,Recall (%)
Attack_type,,,,
ARP_poisioning,1542,1532.0,10.0,99.35
NMAP_UDP_SCAN,551,550.0,1.0,99.82
DDOS_Slowloris,116,NaN,NaN,NaN
DOS_SYN_Hping,17893,NaN,NaN,NaN
Metasploit_Brute_Force_SSH,6,NaN,NaN,NaN
NMAP_FIN_SCAN,4,NaN,NaN,NaN
NMAP_OS_DETECTION,409,NaN,NaN,NaN
NMAP_TCP_scan,210,NaN,NaN,NaN
NMAP_XMAS_TREE_SCAN,451,NaN,NaN,NaN



--- 3. Tráfico Normal clasificado como Ataque (FP) ---


,Total_Casos,Casos_FP,Porcentaje_FP (%)
Attack_type,,,
Thing_Speak,1564,21.0,1.34
Wipro_bulb,47,3.0,6.38
MQTT_Publish,792,NaN,0.00



--- 4. Distribución general de clases (Muestra completa) ---
Las 5 clases con MENOS datos en todo el dataset:
Attack_type
NMAP_FIN_SCAN                   28
Metasploit_Brute_Force_SSH      36
Wipro_bulb                     219
DDOS_Slowloris                 533
NMAP_TCP_scan                 1002
Name: count, dtype: int64
